# New Training period for Audit Model Approval- Oct22 to Mar23


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from scipy import stats
import warnings
import scipy.stats as ss
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from lightgbm import LGBMClassifier
import xgboost as xgb
from xgboost import cv
from hyperopt import fmin, tpe, hp, anneal, Trials
import  gc
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.metrics import precision_score, recall_score, confusion_matrix, f1_score, roc_auc_score, roc_curve, accuracy_score, classification_report, auc
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, auc
warnings.filterwarnings("ignore") 
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
import pickle
import json
import shap
# !pip install deepchecks alibi alibi[tensorflow]

In [ ]:
data = pd.read_csv(r"input_file.psv",delimiter='\t')

In [ ]:
data.head()

In [ ]:
df = data.copy(deep=True)

In [ ]:
# data consider for training is of 5 month
df_new_5m = df.loc[df["papp_dt"].between("2022-10-01", "2023-03-31")]

In [ ]:
df_new_5m.shape

In [ ]:
df_new_5m = df_new_5m.drop(['parent_application_id','papp_dt','bad_def_ever_in9mon_gt0','bflperfiossalary',
                            'applicant_profile_segment','applicant_profile_micro_segment','employer_classification',
                            'cibilscr_bkt','appscr_bkt','segment','age','rejection_codes'],axis=1)

In [ ]:
df_new_5m.shape

In [ ]:
nan_cols = [i for i in df_new_5m.columns if df_new_5m[i].isnull().any()]
print(type(nan_cols))
a = df_new_5m.isnull().mean()*100
print(a[a>50])

In [ ]:
# variables have null values gt 50--drop
null_col_gt50 = ['total_writtenoffamtprincipal','total_settlementamount','total_writtenoffamounttotal',
                 'sum_hca_pl_bajaj_lte6m','total_live_bfl_pl_hca','closed_pl_hca_inl6m','closed_pl_hca_inl12m',
                 'closed_offus_pl_hca_inl6m','closed_offus_pl_hca_inl12m','total_hl_hca','avg_hl_sanc_amt',
                 'opened_hl_hca_inl3m','opened_hl_hca_inl6m','opened_hl_hca_inl12m','total_gl_hca','live_gl_hca_at_acq',
                 'opened_gl_hca_inl3m','opened_gl_hca_inl6m','opened_gl_hca_inl12m','closed_gl_hca_inl12m']

df_new_5m = df_new_5m.drop(null_col_gt50,axis=1)

In [ ]:
#Changing the data type of all the "flag" variables to boolean

flg_var_list = [col for col in df_new_5m.columns if '_flg' in col]

for i in flg_var_list:
    if i in df_new_5m.columns:
        df_new_5m[i] = df_new_5m[i].astype(bool)

In [ ]:
# df_new_5m[['movalue']] = df_new_5m[['movalue']].astype('object')
# df_new_5m[['bad_def_ever_in9mon_gt30']] = df_new_5m[['bad_def_ever_in9mon_gt30']].astype('bool')
# df_new_5m['bad_def_ever_in9mon_gt30'] = df_new_5m['bad_def_ever_in9mon_gt30'].map({True: 1, False: 0})

In [ ]:
df_new_5m['bad_def_ever_in9mon_gt30'].value_counts()

In [ ]:
df_new_5m['bad_def_ever_in9mon_gt30'].value_counts()/len(df_new_5m)*100

In [ ]:
#finding % of null values for each column
percent_missing = df_new_5m.isnull().sum() * 100 / len(df_new_5m)
missing_value_df = pd.DataFrame({'column_name': df_new_5m.columns,
                                 'percent_missing': percent_missing})
# missing_value_df
missing_value_df[percent_missing>0.00]

In [ ]:
nan_cols = [i for i in df_new_5m.columns if df_new_5m[i].isnull().any()]
print(type(nan_cols))
a = df_new_5m.isnull().mean()*100
print(a[a>1])

In [ ]:
#fill null values to -1 and 0:
df_new_5m['dpd_recent_6mon_at_acq_days'].fillna(-1,inplace=True)  
df_new_5m['dpd_recent_9mon_at_acq_days'].fillna(-1,inplace=True)
df_new_5m['dpd_recent_12mon_at_acq_days'].fillna(-1,inplace=True)
df_new_5m['dpd_pl_recent_6mon_at_acq_days'].fillna(-1,inplace=True)
df_new_5m['dpd_pl_recent_9mon_at_acq_days'].fillna(-1,inplace=True)

df_new_5m['hca_all_tl'].fillna(-1,inplace=True)
df_new_5m['balance_all_tl'].fillna(-1,inplace=True)
df_new_5m['credit_limit_all_tl'].fillna(-1,inplace=True)
df_new_5m['hca_all_live_tl'].fillna(-1,inplace=True)
df_new_5m['balance_all_live_tl'].fillna(-1,inplace=True)         
df_new_5m['credit_limit_all_live_tl'].fillna(-1,inplace=True)
df_new_5m['total_pl_hca'].fillna(-1,inplace=True)
df_new_5m['avg_pl_sanc_amt'].fillna(-1,inplace=True)
df_new_5m['total_live_pl_hca'].fillna(-1,inplace=True)
df_new_5m['avg_live_pl_sanc_amt'].fillna(-1,inplace=True)
df_new_5m['opened_pl_hca_inl6m'].fillna(-1,inplace=True)
df_new_5m['opened_pl_hca_inl12m'].fillna(-1,inplace=True)
df_new_5m['opened_offus_pl_hca_inl6m'].fillna(-1,inplace=True)
df_new_5m['opened_offus_pl_hca_inl12m'].fillna(-1,inplace=True)
df_new_5m['total_cc_credit_limit'].fillna(-1,inplace=True)
df_new_5m['total_cc_hca'].fillna(-1,inplace=True)
df_new_5m['total_cc_balance'].fillna(-1,inplace=True)
df_new_5m['total_live_cc_credit_limit'].fillna(-1,inplace=True)
df_new_5m['total_live_cc_hca'].fillna(-1,inplace=True)
df_new_5m['total_live_cc_balance'].fillna(-1,inplace=True)
df_new_5m['cc_util_bal_limit'].fillna(-1,inplace=True)
df_new_5m['bre_obligation_amount'].fillna(-1,inplace=True)
df_new_5m['bfl_eligibility_amount'].fillna(-1,inplace=True)
df_new_5m['applicant_imputed_salary'].fillna(-1,inplace=True)

df_new_5m['amountoverdue_all_tl'].fillna(0,inplace=True)
df_new_5m['amountoverdue_all_live_tl'].fillna(0,inplace=True)

In [ ]:
df_new_5m.head()

In [ ]:
#this line added
df_new_5m.reset_index(drop=True, inplace=True)

In [ ]:
df_new_5m.head()

In [ ]:
# df_new_5m.dtypes

In [ ]:
#describe values
pd.options.display.float_format = "{:.2f}".format
df_new_5m.describe(percentiles = [0.01,0.05,0.10,0.25,0.50,0.75,0.85,0.9,0.95,0.98,0.99,0.995]).T

In [ ]:
nan_cols = [i for i in df_new_5m.columns if df_new_5m[i].isnull().any()]
print(type(nan_cols))

In [ ]:
df_corr_exp = df_new_5m.corr()['bad_def_ever_in9mon_gt30']
df_corr_exp.to_excel(r'C:\Users\deepanshusamdani\OneDrive - Bajaj Finserv Direct Limited\Analysis2024\SOL_RejectionAnalysis\Code\model_for_audit_approval\modelanalysisfiles\model_audit_all_var_target_corr_after_fill.xlsx')

In [ ]:
# Function to plot histogram and KDE for skewness check
def plot_skewness(data, variable):
    plt.figure(figsize=(10, 5))
    
    # Histogram
    plt.subplot(1, 2, 1)
    sns.histplot(data[variable], kde=True, bins=30)
    plt.title(f'Histogram of {variable}')
    
    # KDE Plot
    plt.subplot(1, 2, 2)
    sns.kdeplot(data[variable], shade=True)
    plt.title(f'KDE Plot of {variable}')
    
    plt.tight_layout()
plt.show()

# balance_all_tl

In [ ]:
list_dpd_and_flg_col = list(df_new_5m.columns[[i for i in (df_new_5m.columns.str.startswith('dpd_') |
                                                            df_new_5m.columns.str.endswith('_flg') |
                                                            df_new_5m.columns.str.contains('bad_def_ever_in9mon_gt30') |
                                                            df_new_5m.columns.str.contains('movalue') | 
                                                            df_new_5m.columns.str.contains('appscore') |
                                                            df_new_5m.columns.str.contains('cibilscore')
                                                          )]])

df_dpd_and_flg_col = df_new_5m[list_dpd_and_flg_col]

In [ ]:
df_dpd_and_flg_col.head()

In [ ]:
#considering only those variables for outlier treatment those are numeric variables only, not flag or dpd value varaibles
# only count and sanction amount variables should be treated for outliers
list_for_out_treat = list(df_new_5m.columns[[i for i in ((df_new_5m.columns.str.contains('cnt') | 
                                                          df_new_5m.columns.str.contains('hca') | 
                                                          df_new_5m.columns.str.contains('sanc') | 
                                                          df_new_5m.columns.str.contains('amt') |
                                                          df_new_5m.columns.str.contains('amount') |
                                                          df_new_5m.columns.str.contains('amountoverdue_all_tl') |
                                                          df_new_5m.columns.str.contains('bal') |
                                                          df_new_5m.columns.str.contains('balance') | 
                                                          df_new_5m.columns.str.contains('limit') | 
                                                          df_new_5m.columns.str.contains('salary') |
                                                          df_new_5m.columns.str.contains('sal')) & 
                                                          ((~df_new_5m.columns.str.contains('recent')) &
                                                            (~df_new_5m.columns.str.contains('flg'))))]])

df_for_out_treat = df_new_5m[list_for_out_treat]

# clip the variables on different thresholds so that better correaltion can occured
# manually compared the correlation before the outlier treatment and after outlier treatment and figuredout the threshold capping
# need to write a function for this:
#logic- raw_data_corr > clip_995 then clip those variables at 999; raw_data_corr < clip_995 the 995; else 995

list_of_col_for_999 = ['cnt_pl_enq_l1m','cnt_pl_enq_l2m','cnt_pl_enq_l3m','cnt_pl_enq_l6m','cnt_pl_enq_l9m',
                        'cnt_pl_enq_l12m','cnt_pl_enq_l24m','cnt_hl_enq_l1m','cnt_hl_enq_l2m','cnt_hl_enq_l3m',
                       'cnt_all_unsec_enq_l1m','cnt_all_unsec_enq_l2m','cnt_all_unsec_enq_l3m','cnt_all_unsec_enq_l6m',
                       'cnt_all_unsec_enq_l9m','cnt_all_unsec_enq_l12m','cnt_all_unsec_enq_l24m','cnt_all_sec_enq_l1m',
                       'cnt_all_sec_enq_l2m','cnt_all_sec_enq_l3m','cnt_all_sec_enq_l6m','cnt_all_sec_enq_l9m',
                       'cnt_all_sec_enq_l12m','cnt_all_sec_enq_l24m']

df_for_out_treat_995 = df_for_out_treat.drop(list_of_col_for_999, axis= 1)

df_for_out_treat_999 = df_for_out_treat[list_of_col_for_999]


In [ ]:
def var_clip_995(clip_df):
    col=[]
    lq=[]
    uq=[]
    dict_995 = {}
    
    for i in clip_df:
        ll= clip_df[i].quantile(0.01)
        ul= clip_df[i].quantile(0.995)
        col.append(i)
        lq.append(ll)
        uq.append(ul)
        clip_df[i] = clip_df[i].clip(ll,ul)
        
    dict_995 = {key: [val1, val2] for key, val1, val2 in zip(col, lq, uq)} 
    with open(r"C:\Users\deepanshusamdani\OneDrive - Bajaj Finserv Direct Limited\Analysis2024\SOL_RejectionAnalysis\Code\model_for_audit_approval\pkl_json_files\model_audit_outlier_995_clip.json", "w") as outfile:
        json.dump(dict_995, outfile)
    return  clip_df
    
var_clip_995_select = var_clip_995(df_for_out_treat_995)


def var_clip_999(clip_df):
    col=[]
    lq=[]
    uq=[]
    dict_999 = {}
    
    for i in clip_df:
        ll= clip_df[i].quantile(0.01)
        ul= clip_df[i].quantile(0.999)
        col.append(i)
        lq.append(ll)
        uq.append(ul)
        clip_df[i] = clip_df[i].clip(ll,ul)
        
    dict_999 = {key: [val1, val2] for key, val1, val2 in zip(col, lq, uq)} 
    with open(r"C:\Users\deepanshusamdani\OneDrive - Bajaj Finserv Direct Limited\Analysis2024\SOL_RejectionAnalysis\Code\model_for_audit_approval\pkl_json_files\model_audit_outlier_999_clip.json", "w") as outfile:
        json.dump(dict_999, outfile)
    return  clip_df
    
var_clip_999_select = var_clip_999(df_for_out_treat_999)

In [ ]:
var_clip_995_select.head()

In [ ]:
var_clip_999_select.head()

In [ ]:
var_clip_select = pd.merge(var_clip_995_select,var_clip_999_select, how= 'inner', left_index=True, right_index=True)

In [ ]:
var_clip_select.head()

In [ ]:
pd.options.display.float_format = "{:.2f}".format
var_clip_select.describe(percentiles = [0.01,0.05,0.10,0.25,0.50,0.75,0.85,0.9,0.95,0.98,0.99,0.995,0.997,0.998,0.999]).T

In [ ]:
# Joining outlier variables with core dataframe 
df_final_df =  pd.merge(df_dpd_and_flg_col,var_clip_select, how= 'inner', left_index=True, right_index=True)

In [ ]:
nan_cols = [i for i in var_clip_select.columns if var_clip_select[i].isnull().any()]
print(type(nan_cols))
a = var_clip_select.isnull().mean()*100
print(a[a>0])

In [ ]:
df_final_df.head()

In [ ]:
# df_corr_exp_select = df_final_df.corr()['bad_def_ever_in9mon_gt30']
# df_corr_exp_select.to_excel(r'C:\Users\deepanshusamdani\OneDrive - Bajaj Finserv Direct Limited\Analysis2024\SOL_RejectionAnalysis\Code\model_for_audit_approval\modelanalysisfiles\model_audit_after_outlier_treat_corr.xlsx')

In [ ]:
df_corr_exp_select_1 = df_final_df.corr(method='spearman')['bad_def_ever_in9mon_gt30']
df_corr_exp_select_1.to_excel(r'C:\Users\deepanshusamdani\OneDrive - Bajaj Finserv Direct Limited\Analysis2024\SOL_RejectionAnalysis\Code\model_for_audit_approval\modelanalysisfiles\model_audit_after_outlier_treat_corr_spearman.xlsx')

In [ ]:
#removed appscore varibale from the list 

c_var = ['opened_pl_cnt_inl6m', 'cnt_pl_enq_l6m', 'closed_pl_cnt_inl6m',
       'cnt_all_unsec_enq_l2m', 'total_pl_cnt', 'total_live_pl_cnt',
       'cnt_pl_enq_l1m', 'cnt_pl_enq_ever', 'len_4live_pl_flg',
       'dpd_pl_recent_6mon_at_acq_gte30_flg', 'cnt_cc_enq_l6m',
       'cnt_cd_enq_l3m', 'cnt_all_sec_enq_l6m', 'dpd_recent_6m_gte30_cnt',
       'cnt_cc_enq_l1m', 'cnt_bajaj_pl_enq_l6m',
       'dpd_pl_recent_6mon_at_acq_days', 'cnt_all_sec_enq',
       'wooff_or_settled_gt10k_flg',
       'more_than_once_gte30_dpd_in_recent6m_flg',
       'dpd_recent_6mon_at_acq_days', 'opened_pl_hca_inl6m',
       'lpd_in_gt6m_lte24_pappdt_flg', 'lpd_in_lte6m_pappdt_flg',
       'cc_util_bal_limit', 'total_live_cc_credit_limit', 'avg_pl_sanc_amt',
       'dpd_pl_recent_6mon_at_acq_eq0_flg']


for column in df_final_df.select_dtypes(include=['int64','float64']).columns:
    if column in c_var:
        plot_skewness(df_final_df, column)

In [ ]:
# a = [i for i in df_final_df.columns if i in c_var]

In [ ]:
# print(df_final_df.select_dtypes(include=['int64','float64']).columns.to_list())

In [ ]:
df_final_df.head()

In [ ]:
# df_final_df.dtypes

In [ ]:
# def bivar_plot(dpd_var):
#     print("col: ", dpd_var.columns)
#     for i in range(len(dpd_var.columns)):
#         if i < len(dpd_var.columns)-1:
#             ax = sns.boxplot(x = 'bad_def_ever_in9mon_gt30', y = dpd_var.columns[i], data=dpd_var)
#             plt.show()


In [ ]:
# a = [i for i in df_final_df.select_dtypes(include=['int64','float64']).columns if i in c_var]
# a.append('bad_def_ever_in9mon_gt30')

In [ ]:
df_final_df.head()

In [ ]:
df_final_df.shape

In [ ]:
# bivar_plot(df_final_df[a])

In [ ]:
#WOE and IV calculation
# df_new_5m_selected
def iv_woe(data, target, bins=10, show_woe=False):
    
    #Empty Dataframe
    newDF,woeDF = pd.DataFrame(), pd.DataFrame()
    
    #Extract Column Names
    cols = data.columns
    
    #Run WOE and IV on all the independent variables
    for ivars in cols[~cols.isin([target])]:
        if (data[ivars].dtype.kind in 'bifc') and (len(np.unique(data[ivars]))>10):
            binned_x = pd.qcut(data[ivars], bins,  duplicates='drop')
            d0 = pd.DataFrame({'x': binned_x, 'y': data[target]})
            print("if ivars: ",ivars)
            
        else:
            d0 = pd.DataFrame({'x': data[ivars], 'y': data[target]})
            print("else ivars: ",ivars)
        
        d = d0.groupby("x", as_index=False).agg({"y": ["count", "sum"]})
        d.columns = ['Cutoff', 'N', 'Events']
        d['% of Events'] = np.maximum(d['Events'], 0.5) / d['Events'].sum()
        d['Non-Events'] = d['N'] - d['Events']
        d['% of Non-Events'] = np.maximum(d['Non-Events'], 0.5) / d['Non-Events'].sum()
        d['WoE'] = np.log(d['% of Events']/d['% of Non-Events'])
        d['IV'] = d['WoE'] * (d['% of Events'] - d['% of Non-Events'])
        print("d: ",d)
        print("------------------------------------------------------------------------------------------")
        d.insert(loc=0, column='Variable', value=ivars)
        #print("Information value of " + ivars + " is " + str(round(d['IV'].sum(),6)))
        temp =pd.DataFrame({"Variable" : [ivars], "IV" : [d['IV'].sum()]}, columns = ["Variable", "IV"])
        newDF=pd.concat([newDF,temp], axis=0)
        woeDF=pd.concat([woeDF,d], axis=0)

        #Show WOE Table
        # if show_woe == True:
        #     print(d)
    return newDF, woeDF

In [ ]:
def map_predictive_power(iv_value):
    if iv_value < 0.02:
        return 'Useless'
    elif 0.02 <= iv_value < 0.1:
        return 'Weak'
    elif 0.1 <= iv_value < 0.3:
        return 'Medium'
    elif 0.3 <= iv_value < 0.5:
        return 'Strong'
    elif iv_value >= 0.5:
        return 'Suspiciously good; too good to be true'
    else:
        return 'Unknown'

In [ ]:
iv, woe = iv_woe(data = df_final_df.loc[:,df_final_df.columns!= 'movalue'], target = 'bad_def_ever_in9mon_gt30', bins=10, show_woe = False)

iv['Predictive Power'] = iv['IV'].apply(map_predictive_power)

# Create new columns indicating whether values meet the thresholds
iv['ge_0pt01'] =  iv['IV'] >= 0.01
iv['ge_0pt02'] =  iv['IV'] >= 0.02
iv['ge_0pt025'] = iv['IV'] >= 0.025
iv['ge_0pt05'] =  iv['IV'] >= 0.05


In [ ]:
iv

In [ ]:
df_final_df.groupby('bad_def_ever_in9mon_gt30')['dpd_recent_9mon_at_acq_days'].count()

In [ ]:
# Count of values >= 0.01
count_ge_0_01 = (iv['IV'] >= 0.01).sum()

# Count of values >= 0.02
count_ge_0_02 = (iv['IV'] >= 0.02).sum()

# Count of values >= 0.025
count_ge_0_025 = (iv['IV'] >= 0.025).sum()

# Count of values >= 0.05
count_ge_0_05 = (iv['IV'] >= 0.05).sum()

print(f"Count of values >= 0.01: {count_ge_0_01}")
print(f"Count of values >= 0.02: {count_ge_0_02}")
print(f"Count of values >= 0.025: {count_ge_0_025}")
print(f"Count of values >= 0.05: {count_ge_0_05}")

# picked best correlation and IV var

In [ ]:
#based on correlation and IV, selected variables
df_new_5m_selected = df_final_df[['movalue','dpd_recent_6mon_at_acq_days','dpd_pl_recent_6mon_at_acq_days',
                                  'dpd_pl_recent_9mon_at_acq_days','dpd_pl_recent_6mon_at_acq_eq0_flg',
                                  'dpd_pl_recent_6mon_at_acq_gte30_flg','dpd_pl_recent_9mon_at_acq_gte30_flg',
                                  'dpd_recent_6m_gte30_cnt','more_than_once_gte30_dpd_in_recent6m_flg','cnt_all_sec_enq',
                                  'cnt_all_sec_enq_l3m','cnt_all_sec_enq_l6m','cnt_all_sec_enq_l9m','cnt_all_unsec_enq_l1m',
                                  'cnt_all_unsec_enq_l2m','cnt_all_unsec_enq_l6m','cnt_bajaj_pl_enq_l6m',
                                  'cnt_bajaj_pl_enq_l9m','cnt_cc_enq_l12m','cnt_cc_enq_l1m','cnt_cc_enq_l3m','cnt_cc_enq_l6m',
                                  'cnt_cc_enq_l9m','cnt_cd_enq_l3m','cnt_pl_enq_ever','cnt_pl_enq_l1m','cnt_pl_enq_l3m',
                                  'cnt_pl_enq_l6m','appscore','avg_pl_sanc_amt','cc_util_bal_limit',
                                  'total_live_cc_credit_limit','opened_offus_pl_hca_inl12m','opened_offus_pl_hca_inl6m',
                                  'opened_pl_hca_inl12m','opened_pl_hca_inl6m','wooff_or_settled_gt10k_flg','len_4live_pl_flg',
                                  'lpd_in_gt6m_lte24_pappdt_flg','lpd_in_lte6m_pappdt_flg','closed_pl_cnt_inl12m',
                                  'closed_pl_cnt_inl6m','opened_offus_pl_cnt_inl6m','opened_pl_cnt_inl6m','total_live_pl_cnt',
                                  'total_pl_cnt','bad_def_ever_in9mon_gt30']]

In [ ]:
df_new_5m_selected.shape

In [ ]:
df_new_5m_selected.head()

In [ ]:
pd.options.display.float_format = "{:.2f}".format
df_new_5m_selected.describe(percentiles = [0.01,0.05,0.10,0.25,0.50,0.75,0.85,0.9,0.95,0.98,0.99,0.995,0.997,0.998,0.999]).T

# create different dataframe for set of related var to remove intercorrelation b/w variables

In [ ]:
df_dpd_var_corr = df_new_5m_selected[['dpd_recent_6mon_at_acq_days','dpd_pl_recent_6mon_at_acq_days',
                                      'dpd_pl_recent_9mon_at_acq_days','dpd_pl_recent_6mon_at_acq_eq0_flg',
                                      'dpd_pl_recent_6mon_at_acq_gte30_flg','dpd_pl_recent_9mon_at_acq_gte30_flg',
                                      'dpd_recent_6m_gte30_cnt','more_than_once_gte30_dpd_in_recent6m_flg']]


df_risk_forced_var_corr = df_new_5m_selected[['wooff_or_settled_gt10k_flg','len_4live_pl_flg','lpd_in_gt6m_lte24_pappdt_flg',
                                              'lpd_in_lte6m_pappdt_flg','appscore']]


df_enq_var_corr = df_new_5m_selected[['cnt_all_sec_enq','cnt_all_sec_enq_l3m','cnt_all_sec_enq_l6m','cnt_all_sec_enq_l9m',
                                      'cnt_all_unsec_enq_l1m','cnt_all_unsec_enq_l2m','cnt_all_unsec_enq_l6m',
                                      'cnt_bajaj_pl_enq_l6m','cnt_bajaj_pl_enq_l9m','cnt_cc_enq_l12m','cnt_cc_enq_l1m',
                                      'cnt_cc_enq_l3m','cnt_cc_enq_l6m','cnt_cc_enq_l9m','cnt_cd_enq_l3m','cnt_pl_enq_ever',
                                      'cnt_pl_enq_l1m','cnt_pl_enq_l3m','cnt_pl_enq_l6m']]


df_cnt_amt_var_corr = df_new_5m_selected[['closed_pl_cnt_inl12m','closed_pl_cnt_inl6m','opened_offus_pl_cnt_inl6m',
                                          'opened_pl_cnt_inl6m','total_live_pl_cnt','total_pl_cnt','avg_pl_sanc_amt',
                                          'cc_util_bal_limit','total_live_cc_credit_limit','opened_offus_pl_hca_inl12m',
                                          'opened_offus_pl_hca_inl6m','opened_pl_hca_inl12m','opened_pl_hca_inl6m']]

In [ ]:
# # Create correlation matrix
# corr_matrix = df_dpd_var_corr.corr().abs()
# # Select upper triangle of correlation matrix
# upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# # Find features with correlation greater than 0.95
# to_drop = [column for column in upper.columns if any(upper[column] > 0.61)]
# # Drop features 
# df_dpd_var_corr.drop(to_drop, axis=1, inplace=True)

In [ ]:
# Drop features 
# df_dpd_var_corr.drop(to_drop, axis=1, inplace=True)

In [ ]:
plt.figure(figsize=(16,10))
sns.heatmap(df_dpd_var_corr.corr(),annot=True, fmt='.2g', cmap="viridis",annot_kws={"size":10})
plt.show()

In [ ]:
plt.figure(figsize=(16,10))
sns.heatmap(df_risk_forced_var_corr.corr(),annot=True, fmt='.2g', cmap="viridis",annot_kws={"size":10})
plt.show()

In [ ]:
plt.figure(figsize=(16,10))
sns.heatmap(df_enq_var_corr.corr(),annot=True, fmt='.2g', cmap="viridis",annot_kws={"size":10})
plt.show()

In [ ]:
plt.figure(figsize=(16,10))
sns.heatmap(df_cnt_amt_var_corr.corr(),annot=True, fmt='.2g', cmap="viridis",annot_kws={"size":10})
plt.show()

In [ ]:
#order of columns selected based on the Highest Pos corr -> Lowest Neg corr
df_new_5m_selected_corr_iv = df_new_5m_selected[['movalue','opened_pl_cnt_inl6m','cnt_pl_enq_l6m','closed_pl_cnt_inl6m',
                                                 'cnt_all_unsec_enq_l2m','total_pl_cnt','total_live_pl_cnt',
                                                 'cnt_pl_enq_l1m','cnt_pl_enq_ever','len_4live_pl_flg',
                                                 'dpd_pl_recent_6mon_at_acq_gte30_flg','cnt_cc_enq_l6m',
  'cnt_cd_enq_l3m','cnt_all_sec_enq_l6m','dpd_recent_6m_gte30_cnt','cnt_cc_enq_l1m','cnt_bajaj_pl_enq_l6m',
  'dpd_pl_recent_6mon_at_acq_days','cnt_all_sec_enq','wooff_or_settled_gt10k_flg',
  'dpd_recent_6mon_at_acq_days','opened_pl_hca_inl6m','cc_util_bal_limit','total_live_cc_credit_limit',
                                                 'avg_pl_sanc_amt','dpd_pl_recent_6mon_at_acq_eq0_flg',
                                                 'more_than_once_gte30_dpd_in_recent6m_flg','lpd_in_gt6m_lte24_pappdt_flg','lpd_in_lte6m_pappdt_flg',
                                                 'bad_def_ever_in9mon_gt30']]




In [ ]:
df_new_5m_selected_corr_iv.shape

In [ ]:
df_new_5m_selected_corr_iv.head()

In [ ]:
plt.figure(figsize=(16,10))
sns.heatmap(df_new_5m_selected_corr_iv.corr(),annot=True, fmt='.2g', cmap="viridis",annot_kws={"size":10})
plt.show()

In [ ]:
flg_var_list = [col for col in df_new_5m_selected_corr_iv.columns if '_flg' in col]

for i in flg_var_list:
    if i in df_new_5m_selected_corr_iv.columns:
        df_new_5m_selected_corr_iv[i] = df_new_5m_selected_corr_iv[i].astype('int64')

In [ ]:
df_new_5m_selected_corr_iv.dtypes

In [ ]:
num_cols = list(df_new_5m_selected_corr_iv.select_dtypes(['float64','int64']).columns)

In [ ]:
print(num_cols)

In [ ]:
pd.options.display.float_format = "{:.2f}".format
df_new_5m_selected_corr_iv.describe(percentiles = [0.01,0.05,0.10,0.25,0.50,0.75,0.85,0.9,0.95,0.98,0.99,0.995,0.997,0.998,0.999]).T

# ANOVA statistic

In [ ]:
df_new_5m_selected_corr_iv.head()

In [ ]:
unrelated_num_cols = []
categorical_col = 'bad_def_ever_in9mon_gt30'

for i in num_cols:
    # Perform Kruskal-Wallis test
    grouped_data = [df_new_5m_selected_corr_iv[i][df_new_5m_selected_corr_iv[categorical_col] == category] for category in df_new_5m_selected_corr_iv[categorical_col].unique()]
    statistic, p_value = stats.f_oneway(*grouped_data)

    # Set the significance level (alpha)
    alpha = 0.05

    # Print the results with appropriate text color
    if p_value < alpha:
        print( f"ANOVA statistic: {round(statistic, 2)}")
        print(f"p-value: {p_value}")
        print("\033[32m" + f"Reject the null hypothesis: There is a significant relationship between {i} and {categorical_col}")
        print("\033[0m")  # Reset text color to default
    else:
        print( f"ANOVA statistic: {round(statistic, 2)}")
        print(f"p-value: {p_value}")
        print("\033[31m" + f"No significant relationship between {i} and {categorical_col}")
        print("\033[0m")  # Reset text color to default
        unrelated_num_cols.append(i)

In [ ]:
# df_new_5m_selected_corr_iv.reset_index(drop=True,inplace=True)

In [ ]:
id_cols = ['movalue']

In [ ]:
drop_cols = ['bad_def_ever_in9mon_gt30']

In [ ]:
X = df_new_5m_selected_corr_iv.drop(columns=['bad_def_ever_in9mon_gt30'])
Y = df_new_5m_selected_corr_iv['bad_def_ever_in9mon_gt30']
X_features = X.drop(columns=id_cols)

In [ ]:
X.head()

In [ ]:
X.shape

In [ ]:
Y.head()

In [ ]:
Y.shape

In [ ]:
X_features.head()

In [ ]:
X_train, X_test, Y_train, Y_test, id_train, id_test = train_test_split(X_features, Y, X[id_cols], test_size=0.25, random_state=42,stratify=Y)

In [ ]:
pd.options.display.float_format = "{:.2f}".format
a= X_features.describe(percentiles = [0.01,0.05,0.10,0.25,0.50,0.75,0.85,0.9,0.95,0.98,0.99,0.995,0.997,0.998,0.999]).T

In [ ]:
a

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
Y_train.head()

In [ ]:
Y_test.head()

In [ ]:
print(X_train.shape)
print(Y_train.shape)
print(X_test.shape)
print(Y_test.shape)

In [ ]:
id_train.head()

In [ ]:
id_test.head()

In [ ]:
pd.options.display.float_format = "{:.2f}".format
X_train.describe(percentiles = [0.01,0.05,0.10,0.25,0.50,0.75,0.85,0.9,0.95,0.98,0.99,0.995,0.997,0.998,0.999]).T

In [ ]:
def data_split(df):
    """Split data based on 'TIMEPERIOD' into train and val
    Parameters
    ----------
    df : DataFrame

    Returns
    -------
    train : DataFrame
    val : DataFrame
    """
    # # Convert columns to datetime
    # df['lead_month'] = pd.to_datetime(df['lead_month'])
    # df['lead_month'] = pd.to_datetime(df['lead_month'])

    # # Split the DataFrame based on the 'TIMEPERIOD' column
    # train = df[(df['lead_month'] >= '2024-02-01') & (df['lead_month'] <= '2024-03-01')]
    # val = df[df['lead_month'] == '2024-04-01']
    
    X = df.drop('bad_def_ever_in9mon_gt30',axis=1)
    y = df['bad_def_ever_in9mon_gt30']
    
    
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42,stratify=y)

    X_train_df = pd.DataFrame(X_train)
    X_val_df = pd.DataFrame(X_val)
    y_train_series = pd.Series(y_train)
    y_val_series = pd.Series(y_val)
    # Combine X_train and y_train into a single DataFrame
    train_combined = X_train_df.copy()
    train_combined['bad_def_ever_in9mon_gt30'] = y_train_series.values
    # Combine X_val and y_val into a single DataFrame
    val_combined = X_val_df.copy()
    val_combined['bad_def_ever_in9mon_gt30'] = y_val_series.values

    # Reset index for both DataFrames
    return train_combined.reset_index(drop=True), val_combined.reset_index(drop=True)

In [ ]:
train, val = data_split(df_new_5m_selected_corr_iv)
train_rf, val_rf = data_split(df_new_5m_selected_corr_iv)

In [ ]:
train.shape

# Hyperparameter Tunning of LGBM

In [ ]:
def gini(y_pred, data):
    y_true = data.get_label()
    gini_score = 2 * roc_auc_score(y_true, y_pred) - 1
    return 'gini', gini_score, True

In [ ]:
def gini_normalized(y_true, y_pred):
    return 2 * roc_auc_score(y_true, y_pred) - 1

In [ ]:
result_ho = pd.DataFrame()
i = 0

## Defining Objective Function
## will return the score to optimise
def objective(space):
    
    global i
    global result_ho
    
#     print("train df: ", train.head())
#     print("val df: ", val.head())
    
#     print("train shape: ", train.shape)
#     print("val shape: ", val.shape)
    
    #Creating Lightgbm DataFrame
    lgb_train = lgb.Dataset(train.drop(columns = id_cols+drop_cols), label = train.bad_def_ever_in9mon_gt30)
    lgb_val = lgb.Dataset(val.drop(columns = id_cols+drop_cols), label = val.bad_def_ever_in9mon_gt30)

    
#     boosting_types = ['gbdt', 'goss']

#     if isinstance(space['boosting_type'], int):
#         boosting = boosting_types[space['boosting_type']]  # map index to string
#     else:
#         boosting = space['boosting_type']  # already string
        
    #Parameters of model
    params = {
        'num_leaves': int(space['num_leaves']),
        'max_depth': int(space['max_depth']),
        'learning_rate': space['learning_rate'],
        'objective': 'binary',
        'metric': 'auc',
        "boosting":"gbdt",
        'feature_fraction' : space['feature_fraction'],
        'max_bin' : int(space['max_bin']),
        'min_data_in_leaf': int(space['min_data_in_leaf']),
        "min_data_in_bin": int(space['min_data_in_bin']),
        "bagging_freq" : int(space['bagging_freq']),
        'bagging_fraction': space['bagging_fraction'],
        "lambda_l1": space['lambda_l1'],
        "lambda_l2": space['lambda_l2'],
#         'pos_bagging_fraction' : space['pos_bagging_fraction'],
#         'neg_bagging_fraction' : space['neg_bagging_fraction'],
#         'top_rate' : space['top_rate'],
#         'other_rate' : space['other_rate'],
        'n_estimators' : int(space['n_estimators']),
        "random_seed": 42,
        'verbose': -1,
    }
     
    
    """
    params = {
        'num_leaves': int(space['num_leaves']),
        'max_depth': int(space['max_depth']),
        'learning_rate': space['learning_rate'],
        'objective': 'binary',
        'metric': 'auc',
        "boosting": "gbdt",
        'feature_fraction' : space['feature_fraction'],
        'max_bin' : int(space['max_bin']),
        'min_data_in_leaf': int(space['min_data_in_leaf']),
        "min_data_in_bin": int(space['min_data_in_bin']),
        "bagging_freq" : int(space['bagging_freq']),
        'bagging_fraction': space['bagging_fraction'], 
        'pos_bagging_fraction' : space['pos_bagging_fraction'],
        'neg_bagging_fraction' : space['neg_bagging_fraction'],
        "lambda_l1": space['lambda_l1'],
        "lambda_l2": space['lambda_l2'],
        "random_seed": 42,
        'verbose': -1
    }
    """ 
    evals_result = {}
    
    clf = lgb.train(params, lgb_train, 20000, valid_sets=lgb_val,
                valid_names='val',
                early_stopping_rounds=50,
                verbose_eval=False, evals_result=evals_result)
    gc.collect()
    
    result = pd.DataFrame(clf.params, index=[0])
    
    
    # calculate Gini score

    pred_train = clf.predict(train[clf.feature_name()])
    pred_val = clf.predict(val[clf.feature_name()])
    gini_train = gini_normalized(train.bad_def_ever_in9mon_gt30, pred_train)
    gini_val = gini_normalized(val.bad_def_ever_in9mon_gt30, pred_val)

    
    ## Calculating AUC
    pred_train = clf.predict(train[clf.feature_name()])
    pred_val = clf.predict(val[clf.feature_name()])
    gc.collect()
    train_auc = roc_auc_score(train.bad_def_ever_in9mon_gt30, pred_train)#, num_iteration=clf.best_iteration)
    val_auc = roc_auc_score(val.bad_def_ever_in9mon_gt30, pred_val)#, num_iteration=clf.best_iteration)
    gc.collect()

    score = (abs(train_auc - val_auc) + 1)/((1+val_auc)*(1+val_auc))
    
    result["train_auc"] = train_auc
    result["val_auc"] = val_auc
    result["train_test_diff"] = train_auc - val_auc
    result["n_estimators"] = clf.best_iteration
    result["score"] = score
    result["train_gini"] = gini_train
    result["val_gini"] = gini_val
    result["train_test_gini_diff"] = gini_train - gini_val
    
    del clf
    
    result_ho = pd.concat([result_ho,result],ignore_index=True)
#     result_ho = result_ho.append(result)
    result_ho.to_csv(r'C:\Users\deepanshusamdani\OneDrive - Bajaj Finserv Direct Limited\Analysis2024\SOL_RejectionAnalysis\Code\model_for_audit_approval\modelanalysisfiles\model_audit_sol_mvt_hyperopt_results_LGBM.csv', index=False)
    i = i+1
    
    return (score)

# Document for the Hyperparameter

Q1. Since LGBM is the Bossting Algorithm why should we use the pos_bagging_fraction, neg_bagging_fraction parameters?
- LightGBM (LGBM) is itself a boosting algorithm, but parameters like neg_bagging_fraction and pos_bagging_fraction are used for further control over sampling, especially in class-imbalanced datasets.
    
Q2. Why these parameters exist?
- In imbalanced binary classification, there may be many more negative samples (label = 0) than positive samples (label = 1). To avoid overfitting to the majority class and speed up training, LightGBM lets you bag (sample) from each class separately.

Q3. When to set bagging_fraction and bagging_freq?

✅ Required: bagging_freq > 0
- must set bagging_freq > 0 for any form of bagging (including neg/pos bagging) to be activated.
- If bagging_freq = 0 (the default), bagging is disabled entirely, so your neg_bagging_fraction and pos_bagging_fraction will have no effect.
        
        
🔸 Example Values for bagging_freq:
- bagging_freq = 1 → Bagging happens at every boosting iteration
- bagging_freq = 5 → Bagging is applied once every 5 iterations
- bagging_freq = 10 → Bagging is applied once every 10 iterations


Q4. Difference between the uniform and quniform
- hp.uniform	returns a float (Continuous values)
- hp.quniform	returns a Integer values (discrete values)

Q5.Should I use feature_fraction together with bagging_freq, neg_bagging_fraction, and pos_bagging_fraction?
- Yes, we can use feature_fraction alongside those bagging parameters.
- They control different types of sampling during training, so they complement each other:
    >  - bagging_freq + neg_bagging_fraction + pos_bagging_fraction = row (data) sampling — sampling which training examples (rows) to use in each iteration, with special handling of class imbalance.
    >  - feature_fraction = column (feature) sampling — randomly selects a fraction of features (columns) to use when building each tree.
    >  - Using both is common and helps reduce overfitting and speed up training.
    
 
 Q6. bagging_freq and neg/pos_bagging_fraction when to use in model param? 
 - Use only when the boosting type is not 'goss'. because in 'goss' boosting type these param get ignored

In [ ]:
## Hyperparameter space
## Space is selection of data point from the given distribution
## Distribution is defined for every hyperparameter seperately


space = {
    'num_leaves': hp.quniform('num_leaves', 12, 32, 2), # Uniform integer between 2 and 24 1
    'max_depth': hp.quniform('max_depth', 6, 16, 1), # Uniform integer between 2 and 12
    'learning_rate': hp.uniform('learning_rate', 0.005, 0.015), # Values between 0.005 to 0.015
    'feature_fraction' : hp.uniform('feature_fraction', 0.1, 1), # Values between 0.1 to 1
    'max_bin' : hp.quniform('max_bin', 10, 100, 10), # Values between 10 to 100
    'min_data_in_leaf' : hp.quniform('min_data_in_leaf', 25, 500, 25), # Values between 25 to 1000
    'lambda_l1' : hp.uniform('lambda_l1', 0, 20), #50
    'lambda_l2' : hp.uniform('lambda_l2', 0, 20), #50
    'min_data_in_bin' : hp.quniform('min_data_in_bin', 5, 100, 5),
    'bagging_freq': hp.quniform('bagging_freq', 1, 10, 1), 
    'bagging_fraction': hp.uniform('bagging_fraction', 0.7, 0.95),
#     'pos_bagging_fraction' : hp.uniform('pos_bagging_fraction', 0.1, 1),
#     'neg_bagging_fraction' : hp.uniform('neg_bagging_fraction', 0.1, 1),
#     'top_rate': hp.uniform('top_rate', 0.2, 0.5),  # fraction of large gradients kept
#     'other_rate': hp.uniform('other_rate', 0.1, 0.3),  # fraction of small gradients sampled
    'n_estimators': hp.quniform('n_estimators', 50, 500, 1),
#     'boosting_type': hp.choice('boosting_type', ['gbdt', 'goss'])
    }


In [ ]:
best=fmin(fn = objective, # function to optimize
          space = space, # space from which hyperparameter to be choosen
          algo = tpe.suggest, # optimization algorithm, hyperopt will select its parameters automatically
          max_evals = 50,
          rstate = np.random.default_rng(7)
         )

In [ ]:
print(best)

In [ ]:
hyperopt_results = pd.read_csv(r'sol_mvt_hyperopt_results_LGBM.csv')

In [ ]:
# Find the index of the row with the minimum score
best_param_index = hyperopt_results['score'].idxmin()

# Extract the parameters from this row (excluding the score column if necessary)
# Assuming 'score' is in the last column or specify columns if there are multiple metrics
lgbm_params = hyperopt_results.iloc[best_param_index, :-1].to_dict()

# Display the best parameters
print(lgbm_params)

In [ ]:
lgb_train = lgb.Dataset(train.drop(columns = id_cols+drop_cols), label = train['bad_def_ever_in9mon_gt30'])
lgb_val = lgb.Dataset(val.drop(columns = id_cols+drop_cols), label = val['bad_def_ever_in9mon_gt30'])
evals_result = {}
clf_lgbm = lgb.train(lgbm_params, lgb_train, 20000, valid_sets=lgb_val,
                valid_names='val',
                early_stopping_rounds=50,
                verbose_eval=False, evals_result=evals_result)

In [ ]:
clf_lgbm.best_iteration

In [ ]:
#Predictions from Model on Training, Validation and Hold Out
pred_train = clf_lgbm.predict(train[clf_lgbm.feature_name()])
#print(pred_train)
pred_val = clf_lgbm.predict(val[clf_lgbm.feature_name()])
#pred_hold_out = clf.predict(hold_out[clf.feature_name()])

In [ ]:
pred_train_gini = gini_normalized(train.bad_def_ever_in9mon_gt30, pred_train)
pred_val_gini = gini_normalized(val.bad_def_ever_in9mon_gt30, pred_val)

In [ ]:
print("Train Gini: ", pred_train_gini)
print("Validation Gini: ", pred_val_gini)

In [ ]:
def roc_auc(target_list, pred_list):
    """Print ROC AUC for Target and Predictions
    Parameters
    ----------
    target_list : list
        List of Multiple Target Arrays.
    pred_list : list
        List of Multiple Predicted Array Arrays
    """

    print(roc_auc_score(target_list[0], pred_list[0]))
    print(roc_auc_score(target_list[1], pred_list[1]))
    #print(roc_auc_score(target_list[2], pred_list[2]))

In [ ]:
## PR AUC
def pr_auc(target_list, pred_list):
    """Print PR AUC Values
    Parameters
    ----------
    target_list : list
        List of Multiple Target Arrays.
    pred_list : list
        List of Multiple Predicted Array Arrays
    """

    pr, re, thresholds = precision_recall_curve(target_list[0], pred_list[0])
    pr_val, re_val, thresholds_val = precision_recall_curve(target_list[1], pred_list[1])
    #pr_hold_out, re_hold_out, thresholds_hold_out = precision_recall_curve(target_list[2], pred_list[2])
    print(auc(re, pr))
    print(auc(re_val, pr_val))
    #print(auc(re_hold_out, pr_hold_out))s


In [ ]:
#Printing ROC AUC, PR AUC for Class 1 and Class 0
print("ROC AUC")
roc_auc([train.bad_def_ever_in9mon_gt30,val.bad_def_ever_in9mon_gt30], [pred_train, pred_val])
print("")
print("Class 1 PR AUC")
pr_auc([train.bad_def_ever_in9mon_gt30,val.bad_def_ever_in9mon_gt30], [pred_train, pred_val])
print("")
print("Class 0 PR AUC")
pr_auc([1-train.bad_def_ever_in9mon_gt30,1-val.bad_def_ever_in9mon_gt30], [1-pred_train, 1-pred_val])
print("")

In [ ]:
def roc_auc_curve(target_list, pred_list):
    """Print ROC AUC Curve
    Parameters
    ----------
    target_list : list
        List of Multiple Target Arrays.
    pred_list : list
        List of Multiple Predicted Array Arrays
    """

    fpr, tpr, thresholds = roc_curve(target_list[0], pred_list[0])
    fpr_val, tpr_val, thresholds_test = roc_curve(target_list[1], pred_list[1])
    #fpr_hold_out, tpr_hold_out, thresholds_hold_out = roc_curve(target_list[2], pred_list[2])


    roc_auc = roc_auc_score(target_list[0], pred_list[0])
    roc_auc_val = roc_auc_score(target_list[1], pred_list[1])
    #roc_auc_hold_out = roc_auc_score(target_list[2], pred_list[2])


    plt.figure(figsize=(12, 8))
    plt.grid(True)
    plt.title('ROC Curve')
    plt.plot(fpr, tpr, 'b', label = 'Train AUC = %0.3f' % roc_auc, color = 'C0')
    plt.plot(fpr_val, tpr_val, 'b', label = 'Val AUC = %0.3f' % roc_auc_val, color = 'C1')
    #plt.plot(fpr_hold_out, tpr_hold_out, 'b', label = 'Hold Out AUC = %0.3f' % roc_auc_hold_out, color = 'C2')
    #plt.plot(fpr_true, tpr_true, 'b', label = 'True Values AUC = %0.3f' % roc_auc_oot, color = 'C3')

    plt.legend(loc='best')
    plt.plot([0, 1], [0, 1],'r--', color = 'black')
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.ylabel('True Positive Rate')
    plt.xlabel('False Positive Rate')
    plt.show()

In [ ]:
roc_auc_curve([train.bad_def_ever_in9mon_gt30,val.bad_def_ever_in9mon_gt30], [pred_train, pred_val])

In [ ]:
def pr_auc_curve(target_list, pred_list):
    """Print PR AUC Curve
    Parameters
    ----------
    target_list : list
        List of Multiple Target Arrays.
    pred_list : list
        List of Multiple Predicted Array Arrays
    """

    pr, re, thresholds = precision_recall_curve(target_list[0], pred_list[0])
    pr_val, re_val, thresholds_val = precision_recall_curve(target_list[1], pred_list[1])
   #pr_hold_out, re_hold_out, thresholds_hold_out = precision_recall_curve(target_list[2], pred_list[2])

    precision_score_train = average_precision_score(target_list[0], pred_list[0])
    precision_score_val = average_precision_score(target_list[1], pred_list[1])
    #precision_score_hold_out = average_precision_score(target_list[2], pred_list[2])

    plt.figure(figsize=(12, 8))
    plt.grid(True)
    plt.title('Precision Recall Curve')
    plt.plot(re, pr, 'b', label = 'Train Precision = %0.3f' % precision_score_train, color = 'C0')
    plt.plot(re_val, pr_val,  'b', label = 'Val Precision = %0.3f' % precision_score_val, color = 'C1')
    #plt.plot(re_hold_out, pr_hold_out,  'b', label = 'Hold Out Precision = %0.3f' % precision_score_hold_out, color = 'C2')
    plt.legend(loc='best')
    #plt.plot([0, 1], [0, 1],'r--', color = 'black')
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.ylabel('Precision')
    plt.xlabel('Recall')
    plt.show()


In [ ]:
print("PR Curve for Class 1")
pr_auc_curve([train.bad_def_ever_in9mon_gt30,val.bad_def_ever_in9mon_gt30], [pred_train, pred_val])

In [ ]:
print("PR Curve for Class 0")
pr_auc_curve([1-train.bad_def_ever_in9mon_gt30,1-val.bad_def_ever_in9mon_gt30], [1-pred_train, 1-pred_val])

In [ ]:
feature_importance = pd.DataFrame({"feature":clf_lgbm.feature_name(),"split":clf_lgbm.feature_importance('split'), "gain":clf_lgbm.feature_importance('gain')}).sort_values(by = 'gain', ascending = False)
feature_importance.iloc[:50,:]

# put the features into the highest gain order

In [ ]:
# orderr based on the high to low order of the "gain" value

# based on LGBM Train
df_LGBMClass = df_new_5m_selected_corr_iv[['movalue','opened_pl_cnt_inl6m','cnt_pl_enq_l6m','cnt_all_unsec_enq_l2m','total_live_cc_credit_limit','cc_util_bal_limit','dpd_pl_recent_6mon_at_acq_days','closed_pl_cnt_inl6m','cnt_cc_enq_l6m','avg_pl_sanc_amt','total_pl_cnt','total_live_pl_cnt','dpd_recent_6mon_at_acq_days','cnt_pl_enq_ever','opened_pl_hca_inl6m','dpd_pl_recent_6mon_at_acq_gte30_flg','cnt_pl_enq_l1m','dpd_recent_6m_gte30_cnt','wooff_or_settled_gt10k_flg','cnt_cd_enq_l3m','cnt_all_sec_enq','cnt_cc_enq_l1m','cnt_bajaj_pl_enq_l6m','dpd_pl_recent_6mon_at_acq_eq0_flg','cnt_all_sec_enq_l6m','len_4live_pl_flg','more_than_once_gte30_dpd_in_recent6m_flg','lpd_in_lte6m_pappdt_flg','lpd_in_gt6m_lte24_pappdt_flg','bad_def_ever_in9mon_gt30']]

In [ ]:
df_LGBMClass.shape

In [ ]:
X_lgbmc = df_LGBMClass.drop(columns=['bad_def_ever_in9mon_gt30'])
Y_lgbmc = df_LGBMClass['bad_def_ever_in9mon_gt30']
X_features_lgbmc = X_lgbmc.drop(columns=id_cols)

In [ ]:
X_train_lgbmc, X_test_lgbmc, Y_train_lgbmc, Y_test_lgbmc, id_train_lgbmc, id_test_lgbmc = train_test_split(X_features_lgbmc, Y_lgbmc, X_lgbmc[id_cols], test_size=0.25, random_state=42,stratify=Y_lgbmc)

In [ ]:
print(X_train_lgbmc.shape)
print(X_test_lgbmc.shape)
print(Y_train_lgbmc.shape)
print(Y_test_lgbmc.shape)
print(id_train_lgbmc.shape)
print(id_test_lgbmc.shape)

In [ ]:
# based on LGBM Train
lst_mx = ['opened_pl_cnt_inl6m','cnt_pl_enq_l6m','cnt_all_unsec_enq_l2m','total_live_cc_credit_limit','cc_util_bal_limit','dpd_pl_recent_6mon_at_acq_days','closed_pl_cnt_inl6m','cnt_cc_enq_l6m','avg_pl_sanc_amt','total_pl_cnt','total_live_pl_cnt','dpd_recent_6mon_at_acq_days','cnt_pl_enq_ever','opened_pl_hca_inl6m','dpd_pl_recent_6mon_at_acq_gte30_flg','cnt_pl_enq_l1m','dpd_recent_6m_gte30_cnt','wooff_or_settled_gt10k_flg','cnt_cd_enq_l3m','cnt_all_sec_enq','cnt_cc_enq_l1m','cnt_bajaj_pl_enq_l6m','dpd_pl_recent_6mon_at_acq_eq0_flg','cnt_all_sec_enq_l6m','len_4live_pl_flg','more_than_once_gte30_dpd_in_recent6m_flg','lpd_in_lte6m_pappdt_flg','lpd_in_gt6m_lte24_pappdt_flg']

In [ ]:
mn = MinMaxScaler() 
X_train_lgbmc = pd.DataFrame(mn.fit_transform(X_train_lgbmc),columns=lst_mx,index=X_train_lgbmc.index)
X_test_lgbmc = pd.DataFrame(mn.transform(X_test_lgbmc),columns=lst_mx, index=X_test_lgbmc.index)

In [ ]:
## keep commented until you want to see the va trend
# for column in X_train.select_dtypes(include=['int64','float64']).columns:
#     if column in c_var:
#         plot_skewness(X_train, column)

# loaded scaling/transformation file into pkl file

In [ ]:
pickle.dump(mn, open(r'min_max_scale_json.pkl','wb'))

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
Y_train.head()

In [ ]:
Y_test.head()

# #LGBM KFolds Validation

In [ ]:
scores = []
i=0 
a=[]
e=[]
c=[]
h=[]

lg_val = lgb.LGBMClassifier(
                         class_weight='balanced',
                         boosting_type='gbdt',
                         n_estimators=485,
                         objective='binary',
                         random_state=42,
                         max_depth= 16,
                         num_leaves= 12,
                         max_bin= 20,
                         min_data_in_bin= 65,
                         min_data_in_leaf= 150,
                         bagging_freq = 7,
                         bagging_fraction=  0.7063880699184202,
                         feature_fraction= 0.4582686800375908,
                         lambda_l1= 11.840930510212411,
                         lambda_l2= 15.677069323324568,
                         learning_rate=0.0091655912646359)
''

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# {'num_leaves': 12, 'max_depth': 16, 'learning_rate': 0.0091655912646359, 'objective': 'binary', 'metric': 'auc', 
#  'boosting': 'gbdt', 'feature_fraction': 0.4582686800375908, 'max_bin': 20, 'min_data_in_leaf': 150, 
#  'min_data_in_bin': 65, 'bagging_freq': 7, 'bagging_fraction': 0.7063880699184202, 'lambda_l1': 11.840930510212411, 
#  'lambda_l2': 15.677069323324568, 'random_seed': 42, 'verbose': -1, 'num_iterations': 485, 'early_stopping_round': 50, 
#  'train_auc': 0.6648252462376831, 'val_auc': 0.6497784958681925, 'train_test_diff': 0.0150467503694906, 
#  'n_estimators': 485, 'score': 0.3729363948629969, 'train_gini': 0.3296504924753662, 'val_gini': 0.2995569917363849}

In [ ]:
print(lg_val.get_params)

# to check the model stability on each fold- by iter on the same training data

In [ ]:
for train_index, test_index in cv.split(X_features_lgbmc,Y_lgbmc):
#     Y = pd.DataFrame(Y,columns=['bad_def_ever_in9mon_gt30']) if y_train used as datafrmae then use this 
    i=i+1
    
    print(f"\nFold {i} ", end=' : ')
    print("train_index: ", train_index)
    x_train, x_test, y_train, y_test = X_features_lgbmc.iloc[train_index,:],X_features_lgbmc.iloc[test_index,:], Y_lgbmc.iloc[train_index], Y_lgbmc.iloc[test_index]
    
#     lg_val.fit(x_train,y_train['bad_def_ever_in9mon_gt30'].astype(int),eval_set=[(x_test,y_test),(x_train,y_train)],early_stopping_rounds=20,verbose=False,eval_metric='logloss')
#     lg_val.fit(x_train,y_train.astype(int),eval_set=[(x_test,y_test),(x_train,y_train)],early_stopping_rounds=20,verbose=False,eval_metric='logloss')

    lg_val.fit(x_train,y_train.astype(int),eval_set=[(x_test,y_test),(x_train,y_train)],early_stopping_rounds=50,verbose=-1,eval_metric='logloss')
    
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
    ytrainpred = lg_val.predict(x_train)
    
    k = accuracy_score(y_train.to_numpy().astype(int),ytrainpred)
    c.append(k)
    
    ypred = lg_val.predict(x_test)
    g = accuracy_score(y_test.to_numpy().astype(int),ypred)
    h.append(g)
    
    print(confusion_matrix(y_test.to_numpy().astype(int),ypred))
    print(classification_report(y_test.to_numpy().astype(int),ypred))
    print('Train_Accuracy: '+str(np.round(k, 7)))
    print('Test_Accuracy: '+str(np.round(g, 7)))
    
print(f'\nTrain_Accuracy Mean: {np.round(np.mean(c), 5)} Std: {np.round(np.std(c), 5)}')
print(f'\nTest_Accuracy Mean: {np.round(np.mean(h), 5)} Std: {np.round(np.std(h), 5)}') 

# # LGBM Model

In [ ]:
import lightgbm as lgb

lg = lgb.LGBMClassifier(
                         class_weight='balanced',
                         boosting_type='gbdt',
                         n_estimators=485,
                         objective='binary',
                         random_state=42,
                         max_depth= 16,
                         num_leaves= 12,
                         max_bin= 20,
                         min_data_in_bin= 65,
                         min_data_in_leaf= 150,
                         bagging_freq = 7,
                         bagging_fraction=  0.7063880699184202,
                         feature_fraction= 0.4582686800375908,
                         lambda_l1= 11.840930510212411,
                         lambda_l2= 15.677069323324568,
                         learning_rate=0.0091655912646359)

lg.fit(X_train_lgbmc,Y_train_lgbmc.astype(int),eval_set=[(X_test_lgbmc,Y_test_lgbmc),(X_train_lgbmc,Y_train_lgbmc)],early_stopping_rounds=50,verbose=-1,eval_metric='logloss')

# model loaded

In [ ]:
pickle.dump(lg, open(r'model_lgbm_json.pkl','wb'))

In [ ]:
# Access the booster object
booster = lg._Booster

# Get feature importance
importance_df = pd.DataFrame({
    "feature": booster.feature_name(),
    "split": booster.feature_importance(importance_type='split'),
    "gain": booster.feature_importance(importance_type='gain')
})

In [ ]:
importance_df_sorted = importance_df.sort_values(by='gain', ascending=False)

In [ ]:
# Show top 20 features
top_features = importance_df_sorted.head(35)
print(top_features)

In [ ]:
pred=pd.Series(lg.predict(X_test_lgbmc),index=X_test_lgbmc.index)
pred_prob=lg.predict_proba(X_test_lgbmc)
pred_prob=pd.Series(pred_prob[:,1],index=X_test_lgbmc.index)

In [ ]:
# pred_prob = pred_prob.round(2) 
# (this line can be added here, to fix the Actual Bad rate rank order hold in the decile)
# this has been identified during the Model Docs code creation--- so has been added in this code (SOL_MVT_model_Code_Audit_docs.ipynb) for Audit prespective.
# here after adding this line, prob range has been same as used in OOT, so no issue.

In [ ]:
print(classification_report(Y_test_lgbmc, pred))

In [ ]:
# Instead of resetting the index of y_test, we'll create a DataFrame to link with X_test
dec_test = pd.DataFrame({'bad_def_ever_in9mon_gt30': Y_test_lgbmc.values, 
                          'predicted': pred,
                          'pred_prob': pred_prob}, index=Y_test_lgbmc.index)

In [ ]:
dec_test.head()

In [ ]:
df_dec_test = dec_test.sort_values(by='pred_prob', ascending=False)
df_dec_test['row_id'] = range(len(df_dec_test))
df_dec_test['decile'] = (df_dec_test['row_id'] / (len(df_dec_test)/10)).astype(int)
df_dec_test.loc[df_dec_test['decile'] == 10]=9

In [ ]:
df_dec_test.head()

In [ ]:
min(df_dec_test.index), max(df_dec_test.index)

In [ ]:
#to fix the decile wise min max prob band for OOT data (using this bands in OOT data for deciling)
df_dec_test.groupby(['decile'])["pred_prob"].agg([min,max,"count"]).reset_index()

In [ ]:
#create gains table
gains = df_dec_test.groupby('decile')['bad_def_ever_in9mon_gt30'].agg(['count','sum'])
gains.columns = ['count','actual']
gains
gains['actual'] = gains['actual'].astype(int)
gains['non_actual'] = np.subtract(gains['count'],gains['actual'])
gains['cum_count'] = gains['count'].cumsum()
gains['cum_actual'] = gains['actual'].cumsum()
gains['cum_non_actual'] = gains['non_actual'].cumsum()
gains['percent_cum_actual'] = (gains['cum_actual'] / np.max(gains['cum_actual'])).round(5)
gains['percent_cum_non_actual'] = (gains['cum_non_actual'] / np.max(gains['cum_non_actual'])).round(5)
gains['if_random'] = np.max(gains['cum_actual']) /10
gains['if_random'] = gains['if_random'].cumsum()
gains['lift'] = (gains['cum_actual'] / gains['if_random']).round(2)
gains['K_S'] = (np.abs(np.subtract(gains['percent_cum_actual'],gains['percent_cum_non_actual'])).round(5)) * 100
gains['gain']=(gains['cum_actual']/gains['cum_count']*100).round(2)
gains = pd.DataFrame(gains) 

In [ ]:
gains

In [ ]:
#pred = model.predict(x_test)
cf = confusion_matrix(Y_test_lgbmc.to_numpy().astype(int),pred)

TN,FP = cf[0][0],cf[0][1]
FN,TP = cf[1][0],cf[1][1]

print(cf)
Accuracy = (TP+TN)/(TP+TN+FN+FP)
Recall = TP/(TP+FN)
Specificity = TN/(TN+FP)
Precision = TP/(TP+FP)
F1_Score = 2*((Precision*Recall)/(Precision+Recall))
print('Accuracy: ','{:0.2f}'.format(Accuracy)+'%')
print('Precision: ','{:0.2f}'.format(Precision)+'%')
print('Recall: ','{:0.2f}'.format(Recall)+'%')
print('Specificity: ','{:0.2f}'.format(Specificity)+'%')
print('F1_Score: ','{:0.2f}'.format(F1_Score)+'%') 

In [ ]:
def gini_lgmb_class(y_pred, data):
    y_true = data.get_label()
    gini_score = 2 * roc_auc_score(y_true, y_pred) - 1
    return 'gini', gini_score, True

def gini_normalized_lgmb_class(y_true, y_pred):
    return 2 * roc_auc_score(y_true, y_pred) - 1

In [ ]:
# 3. Predict probabilities
pred_train_1 = lg.predict_proba(X_train_lgbmc)[:, 1]
pred_val_1 = lg.predict_proba(X_test_lgbmc)[:, 1]

In [ ]:
# 4. Compute Gini
train_gini_1 = gini_normalized_lgmb_class(Y_train_lgbmc, pred_train_1)
val_gini_1 = gini_normalized_lgmb_class(Y_test_lgbmc, pred_val_1)

In [ ]:
print("Train Gini:", train_gini_1)
print("Validation Gini:", val_gini_1)

# pred and pred_prob on train data

In [ ]:
pred_train=pd.Series(lg.predict(X_train_lgbmc),index=X_train_lgbmc.index)
pred_prob_train=lg.predict_proba(X_train_lgbmc)
pred_prob_train=pd.Series(pred_prob_train[:,1],index=X_train_lgbmc.index)

In [ ]:
pred_prob_train = pred_prob_train.round(2)

In [ ]:
print(classification_report(Y_train_lgbmc, pred_train))

In [ ]:
dec_train = pd.DataFrame({'bad_def_ever_in9mon_gt30': Y_train_lgbmc.values, 
                          'predicted': pred_train,
                          'pred_prob': pred_prob_train}, index=Y_train_lgbmc.index)

In [ ]:
df_dec_train = dec_train.sort_values(by='pred_prob', ascending=False)
df_dec_train['row_id'] = range(len(df_dec_train))
df_dec_train['decile'] = (df_dec_train['row_id'] / (len(df_dec_train)/10)).astype(int)
df_dec_train.loc[df_dec_train['decile'] == 10]=9

In [ ]:
#create gains table
gains_train = df_dec_train.groupby('decile')['bad_def_ever_in9mon_gt30'].agg(['count','sum'])
gains_train.columns = ['count','actual']
gains_train
gains_train['actual'] = gains_train['actual'].astype(int)
gains_train['non_actual'] = np.subtract(gains_train['count'],gains_train['actual'])
gains_train['cum_count'] = gains_train['count'].cumsum()
gains_train['cum_actual'] = gains_train['actual'].cumsum()
gains_train['cum_non_actual'] = gains_train['non_actual'].cumsum()
gains_train['percent_cum_actual'] = (gains_train['cum_actual'] / np.max(gains_train['cum_actual'])).round(5)
gains_train['percent_cum_non_actual'] = (gains_train['cum_non_actual'] / np.max(gains_train['cum_non_actual'])).round(5)
gains_train['if_random'] = np.max(gains_train['cum_actual']) /10
gains_train['if_random'] = gains_train['if_random'].cumsum()
gains_train['lift'] = (gains_train['cum_actual'] / gains_train['if_random']).round(2)
gains_train['K_S'] = (np.abs(np.subtract(gains_train['percent_cum_actual'],gains_train['percent_cum_non_actual'])).round(5))* 100
gains_train['gain']=(gains_train['cum_actual']/gains_train['cum_count']*100).round(2)
gains_train = pd.DataFrame(gains_train) 

In [ ]:
gains_train

In [ ]:
#pred = model.predict(x_test)
cf_1 = confusion_matrix(Y_train_lgbmc.to_numpy().astype(int),pred_train)

TN1,FP1 = cf_1[0][0],cf_1[0][1]
FN1,TP1 = cf_1[1][0],cf_1[1][1]

print(cf_1)
Accuracy_1 = (TP1+TN1)/(TP1+TN1+FN1+FP1)
Recall_1 = TP1/(TP1+FN1)
Specificity_1 = TN1/(TN1+FP1)
Precision_1 = TP1/(TP1+FP1)
F1_Score_1 = 2*((Precision_1*Recall_1)/(Precision_1+Recall_1))
print('Accuracy: ','{:0.2f}'.format(Accuracy_1)+'%')
print('Precision: ','{:0.2f}'.format(Precision_1)+'%')
print('Recall: ','{:0.2f}'.format(Recall_1)+'%')
print('Specificity: ','{:0.2f}'.format(Specificity_1)+'%')
print('F1_Score: ','{:0.2f}'.format(F1_Score_1)+'%') 

In [ ]:
X_test.head()

In [ ]:
min(X_test.index), max(X_test.index)

In [ ]:
X_test= X_test.join(id_test)

In [ ]:
X_test.head()

In [ ]:
result_test = X_test.join(df_dec_test[['bad_def_ever_in9mon_gt30', 'predicted', 'pred_prob', 'decile']])

In [ ]:
result_test.head(20)

# **Conclusion of the LGBM model**-
1.The Gain is better by using Hyperparameters and the top 15 features are same with or without tunning just the order of importance of the features is change.
2.Without using tunning the importance of the features drop down to zero, but using tunning the low imporatance features also have significant value.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from io import BytesIO
from openpyxl import Workbook
from openpyxl.drawing.image import Image

# Assuming BIVARIATE_TRAIN is already loaded in memory

# Define a function to sort bins with '-inf' first and '+inf' last
def bin_sort_key(bin_label):
    try:
        if isinstance(bin_label, str):
            if '-inf' in bin_label:
                return float('-inf')
            elif '+inf' in bin_label:
                return float('inf')
            else:
                return float(bin_label.strip('[]()').split(',')[0].replace('inf', ''))
        return float('inf')
    except:
        return float('inf')

# Calculate %pop and bad_rate within each variable group
BIVARIATE_TRAIN = BIVARIATE_TRAIN.groupby('variable').apply(
    lambda group: group.assign(
        pop=group['df1_count'] / group['df1_count'].sum(),
        bad_rate=group['df1_target1'] / group['df1_count']
    )
).reset_index(drop=True)

# Sort bins within each variable group
BIVARIATE_TRAIN['bin_sort'] = BIVARIATE_TRAIN['bin'].apply(bin_sort_key)
BIVARIATE_TRAIN = BIVARIATE_TRAIN.sort_values(by=['variable', 'bin_sort'])

# Create Excel workbook and sheet
wb = Workbook()
ws = wb.active
ws.title = "BIVARIATE_TRAIN_CHART"

# Generate and embed plots
row_position = 1
for variable in BIVARIATE_TRAIN['variable'].unique():
    sub_df = BIVARIATE_TRAIN[BIVARIATE_TRAIN['variable'] == variable]
    fig, ax1 = plt.subplots(figsize=(18, 3))

    ax2 = ax1.twinx()
    ax1.bar(sub_df['bin'], sub_df['pop'], color='skyblue', label='%pop')
    ax2.plot(sub_df['bin'], sub_df['bad_rate'], color='red', marker='o', label='bad_rate')

    ax1.set_xlabel('Bin')
    ax1.set_ylabel('%pop', color='skyblue')
    ax2.set_ylabel('bad_rate', color='red')
    plt.title(variable)
    fig.tight_layout()

    # Save plot to BytesIO
    img_data = BytesIO()
    plt.savefig(img_data, format='png')
    plt.close(fig)
    img_data.seek(0)

    # Embed image in Excel
    img = Image(img_data)
    img.anchor = f'A{row_position}'
    ws.add_image(img)
    row_position += 20

# Save workbook
excel_path = "bivariate_chart_train.xlsx"
wb.save(excel_path)
print("Excel file with bivariate charts saved as bivariate_chart_train_only.xlsx")